In [2]:
import pandas as pd
fact_sales_fe=pd.read_csv(r"C:\Users\RohanS\Documents\data retail processed\fact_sales_fe.csv")
monthly_demand = (
    fact_sales_fe
    .groupby(['product_id','order_month'])
    .agg(
        units_sold=('order_item_id','count')
    )
    .reset_index()
)
monthly_demand.to_csv(
    r"C:\Users\RohanS\Documents\data retail processed\montly_demand.csv",
    index=False)
monthly_demand


,product_id,order_month,units_sold
0,00066f42aeeb9f3007548bb9d3f33c38,May,1
1,00088930e925c41fd95ebfe695fd2655,December,1
2,0009406fd7479715e4bef61dd91f2462,December,1
3,000b8f95fcb9e0096488278317764d19,August,2
4,000d9be29b5207b54e86aa1b1ac54872,April,1
...,...,...,...
60791,fff9553ac224cec9d15d49f5a263411f,October,1
60792,fffdb2d0ec8d6a61f0a0a0db3f25b441,April,2
60793,fffdb2d0ec8d6a61f0a0a0db3f25b441,August,2
60794,fffdb2d0ec8d6a61f0a0a0db3f25b441,May,1


In [12]:
inventory_base = (
    monthly_demand
    .groupby('product_id')
    .agg(
        avg_monthly_demand=('units_sold','mean'),
        max_monthly_demand=('units_sold','max'),
        min_monthly_demand=('units_sold','min'),
        demand_std=('units_sold','std')
    )
    .reset_index()
)
inventory_base['demand_std'] = (
    inventory_base['demand_std']
    .fillna(0)
)
inventory_base

,product_id,avg_monthly_demand,max_monthly_demand,min_monthly_demand,demand_std
0,00066f42aeeb9f3007548bb9d3f33c38,1.000000,1,1,0.00000
1,00088930e925c41fd95ebfe695fd2655,1.000000,1,1,0.00000
2,0009406fd7479715e4bef61dd91f2462,1.000000,1,1,0.00000
3,000b8f95fcb9e0096488278317764d19,2.000000,2,2,0.00000
4,000d9be29b5207b54e86aa1b1ac54872,1.000000,1,1,0.00000
...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,1.000000,1,1,0.00000
32947,fff81cc3158d2725c0655ab9ba0f712c,1.000000,1,1,0.00000
32948,fff9553ac224cec9d15d49f5a263411f,1.000000,1,1,0.00000
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,1.666667,2,1,0.57735


In [13]:
import numpy as np
lead_time_months = 7/30

inventory_base['safety_stock'] = (
    1.65 *
    inventory_base['demand_std'] *
    np.sqrt(lead_time_months)
).round().astype(int)
inventory_base["safety_stock"] = (
    inventory_base["safety_stock"]
    .clip(lower=1)
)
inventory_base['reorder_level'] = (
    inventory_base['avg_monthly_demand'] *
    lead_time_months
    +
    inventory_base['safety_stock']
).round().astype(int)
np.random.seed(42)

inventory_base['current_stock'] = (
    inventory_base['avg_monthly_demand'] *
    np.random.uniform(1,3,len(inventory_base))
).round().astype(int)
inventory_base['inventory_status'] = np.where(
    inventory_base['current_stock'] <= inventory_base['reorder_level'],
    'Reorder',
    'Healthy'
)
print(inventory_base)
inventory_base.to_csv(
    r"C:\Users\RohanS\Documents\data retail processed\inventory_bases.csv",
    index=False)


                             product_id  avg_monthly_demand  \
0      00066f42aeeb9f3007548bb9d3f33c38            1.000000   
1      00088930e925c41fd95ebfe695fd2655            1.000000   
2      0009406fd7479715e4bef61dd91f2462            1.000000   
3      000b8f95fcb9e0096488278317764d19            2.000000   
4      000d9be29b5207b54e86aa1b1ac54872            1.000000   
...                                 ...                 ...   
32946  fff6177642830a9a94a0f2cba5e476d1            1.000000   
32947  fff81cc3158d2725c0655ab9ba0f712c            1.000000   
32948  fff9553ac224cec9d15d49f5a263411f            1.000000   
32949  fffdb2d0ec8d6a61f0a0a0db3f25b441            1.666667   
32950  fffe9eeff12fcbd74a2f2b007dde0c58            1.000000   

       max_monthly_demand  min_monthly_demand  demand_std  safety_stock  \
0                       1                   1     0.00000             1   
1                       1                   1     0.00000             1   
2                 

In [14]:
import pandas as pd

warehouse_master = pd.DataFrame({
    "warehouse_id":[
        "WH001",
        "WH002",
        "WH003",
        "WH004",
        "WH005"
    ],

    "warehouse_name":[
        "Sao Paulo DC",
        "Curitiba DC",
        "Salvador DC",
        "Goiania DC",
        "Recife DC"
    ],

    "city":[
        "Sao Paulo",
        "Curitiba",
        "Salvador",
        "Goiania",
        "Recife"
    ],

    "state":[
        "SP",
        "PR",
        "BA",
        "GO",
        "PE"
    ],

    "region":[
        "Southeast",
        "South",
        "Northeast",
        "Central-West",
        "Northeast"
    ],

    "capacity_units":[
        150000,
        90000,
        80000,
        70000,
        60000
    ]
})

In [15]:
warehouse_map = {

"SP":"WH001",
"RJ":"WH001",
"MG":"WH001",
"ES":"WH001",

"PR":"WH002",
"SC":"WH002",
"RS":"WH002",

"BA":"WH003",
"PE":"WH005",
"CE":"WH005",
"PB":"WH005",
"RN":"WH005",
"AL":"WH003",
"SE":"WH003",

"GO":"WH004",
"DF":"WH004",
"MT":"WH004",
"MS":"WH004",

"AM":"WH005",
"PA":"WH005",
"RO":"WH004",
"AC":"WH004",
"RR":"WH005",
"AP":"WH005",
"TO":"WH004",

"MA":"WH005",
"PI":"WH005"
}

In [16]:
seller_mart=pd.read_csv(r"C:\Users\RohanS\Documents\data retail processed\seller_mart.csv")
fact_sales=pd.read_csv(r"C:\Users\RohanS\Documents\data retail\fact_sales.csv")

In [37]:
seller_mapping = seller_mart.copy()
seller_mapping = (
    fact_sales[
        ["seller_id","seller_state"]
    ]
    .drop_duplicates()
)
seller_mapping["warehouse_id"] = (
    seller_mapping["seller_state"]
    .map(warehouse_map)
)
seller_mart["avg_delivery"]=pd.to_timedelta(seller_mart["avg_delivery"])


In [38]:
warehouse_perf = seller_mapping.merge(
    seller_mart,
    on="seller_id",
    how="left")

warehouse_perf = (
    warehouse_perf
    .groupby("warehouse_id")
    .agg(
        total_revenue=("total_revenue", "sum"),
        total_orders=("total_orders", "nunique"),
        sellers=("seller_id", "nunique"),
        avg_review=("avg_review", "mean"),
        avg_delivery=("avg_delivery","mean")
    )
    .reset_index()
)


warehouse_perf = warehouse_perf.merge(
    warehouse_master,
    on="warehouse_id",
    how="left"
)
product_warehouse = (
    fact_sales[
        ["product_id","seller_id"]
    ]
    .drop_duplicates()
)
product_warehouse = product_warehouse.merge(
    seller_mapping,
    on="seller_id",
    how="left"
)
warehouse_inventory = (
    inventory_base.merge(
        product_warehouse,
        on="product_id",
        how="left"
    )
)
warehouse_inventory = (
    warehouse_inventory
    .groupby("warehouse_id")
    .agg(
        total_inventory=("current_stock","sum"),
        products=("product_id","nunique"),
        )
    
    .reset_index()
)
warehouse_perf = warehouse_perf.merge(
    warehouse_inventory,
    on="warehouse_id",
    how="left"
)
warehouse_perf["capacity_utilization"] = (
    warehouse_perf["total_inventory"] /
    warehouse_perf["capacity_units"] * 100
).round(2)




In [40]:
warehouse_perf.to_csv(
    r"C:\Users\RohanS\Documents\data retail processed\warehouse_perf.csv",
    index=False)


In [39]:
warehouse_perf

,warehouse_id,total_revenue,total_orders,sellers,avg_review,avg_delivery,warehouse_name,city,state,region,capacity_units,total_inventory,products,capacity_utilization
0,WH001,11159705.19,231,2287,3.956099,11 days 21:41:46.311698205,Sao Paulo DC,Sao Paulo,SP,Southeast,150000,87394,27088,58.26
1,WH002,2391285.74,101,668,4.024481,12 days 13:09:26.968420833,Curitiba DC,Curitiba,PR,South,90000,16192,5194,17.99
2,WH003,322873.89,13,21,3.955427,15 days 12:44:10.857819723,Salvador DC,Salvador,BA,Northeast,80000,707,134,0.88
3,WH004,209538.63,33,82,4.050472,13 days 18:33:34.417196284,Goiania DC,Goiania,GO,Central-West,70000,2084,720,2.98
4,WH005,190296.20,16,37,3.914290,16 days 03:45:04.523552089,Recife DC,Recife,PE,Northeast,60000,642,194,1.07
